# 航空部件非计划拆换预测 PoC

目标：未来 30 个飞行循环内的非计划拆换。主要门槛为事件级召回率 ≥ 80%，逐轮及总体警报率 ≤ 5%。本 Notebook 只读取冻结数据与实验产物，不参与阈值或模型选择。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.splitter import create_joint_split_manifest, entity_group_diagnostics

DATA_DIR = Path('data/generated/mock_v0_2_0')
data = pd.read_parquet(DATA_DIR / 'mro_dataset.parquet')
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
manifest['summary']

## EDA：正例率、缺失、异常和实体/事件规模

In [ ]:
eda = pd.DataFrame({
    'missing_rate': data[['vibration_rms', 'egt_margin', 'sensor_temperature', 'sensor_pressure', 'sensor_current']].isna().mean(),
})
display(eda)
data['unscheduled_removal_within_30_cycles'].value_counts(dropna=False, normalize=True)

## 联合时间/飞机/部件划分与泄漏证据

In [ ]:
split = create_joint_split_manifest(
    data, train_end='2024-09-01', validation_end='2025-09-01', calibration_end='2025-03-01'
)
display(split.groupby(['outer_partition', 'final_use']).size())
display(entity_group_diagnostics(split).head())

## PR 曲线、校准与阈值分析

训练流程完成后，此处读取 `artifacts/` 中冻结的折外验证预测、校准器、阈值和测试报告。测试数据不得用于模型或策略选择。

## SHAP 与高风险解释卡

LightGBM 使用原始 log-odds 归因并检查加和关系；展示时单独报告校准概率。至少包含两个高风险部件，并补充误报和漏报案例。

## 生产闭环与局限

航后数据接入 → 可用时间过滤 → 时点特征 → 风险评分与 SHAP → 重复抑制及逐轮容量 → AMOS/Maximo 工单队列 → 30 循环延迟标签回流 → 漂移监控、审批重训练与回滚。合成数据结果不代表真实机队性能。